# Create DeLong Table

Compares all preop and combined models within each group and with the other group

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Pair-wise AUC comparison with a correct implementation of DeLong’s test
(Modified for improved formatting and output)
"""

import os
import html
import numpy as np
import pandas as pd
import scipy.stats
from scipy.stats import norm
from sklearn.metrics import roc_auc_score
from IPython.display import display, HTML

# =============================================================================
# SCRIPT CONFIGURATION
# =============================================================================
RESULTS_DIR = '/home/server/Projects/data/AKI/results/'
OUTPUT_CSV_FILE = 'delong_comparison_results.csv'

MODELS_TO_EXCLUDE = [
    'preop_ASA Rule',
    'combined_ASA Rule',
    'combined_Hybrid (MLP + LSTM)'
]

# --- MODIFICATION: Shortened model names ---
model_translate = {
    'log_reg': 'LR', 'xgb': 'GBT',
    'svm_linear': 'SVM (Linear)', 'mlp': 'MLP', 'rf': 'RF',
    'knn': 'KNN', 'asa_rule': 'ASA Rule', 'autogluon': 'AG',
    'lstm': 'LSTM', 'hybrid': 'Hybrid (MLP + LSTM)'
}

model_base_map = {'LSTM': 'base_54k', 'Hybrid (MLP + LSTM)': 'base_54k'}
DEFAULT_BASE = 'base'

# =============================================================================
# FAST & MEMORY-SAFE DELONG (no big matrices)
# =============================================================================
def _structural_components(y_true: np.ndarray,
                           scores:   np.ndarray):
    """
    Return (auc, v10, v01) without building an n_pos × n_neg table.

    v10[i]  = mean_j θ( s_pos[i] , s_neg[j] )
    v01[j]  = mean_i θ( s_pos[i] , s_neg[j] ),      θ = 1, 0.5, 0
    """
    y_true  = y_true.astype(int).ravel()
    scores  = scores.ravel()
    pos     = scores[y_true == 1]
    neg     = scores[y_true == 0]

    n_pos   = pos.size
    n_neg   = neg.size
    if n_pos == 0 or n_neg == 0:
        raise ValueError("Need at least one positive and one negative sample.")

    # Pre-sort once for binary-search look-ups
    neg_sorted = np.sort(neg)
    pos_sorted = np.sort(pos)

    # --- V10: for every positive score, compare with all negatives -----------
    left  = np.searchsorted(neg_sorted, pos, side='left')    # #neg < s_pos
    right = np.searchsorted(neg_sorted, pos, side='right')   # #neg ≤ s_pos
    ties  = right - left
    v10   = (left + 0.5 * ties) / n_neg                      # shape (n_pos,)

    # --- V01: for every negative score, compare with all positives -----------
    left  = np.searchsorted(pos_sorted, neg, side='left')    # #pos < s_neg
    right = np.searchsorted(pos_sorted, neg, side='right')   # #pos ≤ s_neg
    ties  = right - left
    greater = n_pos - right
    v01   = (greater + 0.5 * ties) / n_pos                   # shape (n_neg,)

    auc = v10.mean()                                         # identical to roc_auc_score
    return auc, v10, v01


def delong_test(y_true:  np.ndarray,
                scores1: np.ndarray,
                scores2: np.ndarray):
    """
    DeLong’s test for correlated AUCs (two-sided).
    Returns (auc1, auc2, p_value).
    Memory complexity: O(n) ;  never builds n_pos × n_neg arrays.
    """
    auc1, v10_1, v01_1 = _structural_components(y_true, scores1)
    auc2, v10_2, v01_2 = _structural_components(y_true, scores2)

    n_pos = v10_1.size
    n_neg = v01_1.size

    s10 = np.cov(np.vstack([v10_1, v10_2]), ddof=1)
    s01 = np.cov(np.vstack([v01_1, v01_2]), ddof=1)

    var_auc_diff = (s10[0, 0] + s10[1, 1] - 2 * s10[0, 1]) / n_pos + \
                   (s01[0, 0] + s01[1, 1] - 2 * s01[0, 1]) / n_neg

    if var_auc_diff <= 0 or not np.isfinite(var_auc_diff):
        return auc1, auc2, 1.0          # numerical safeguard

    z = (auc1 - auc2) / np.sqrt(var_auc_diff)
    p = 2 * (1 - norm.cdf(abs(z)))
    return auc1, auc2, p

# =============================================================================
# DATA PREPARATION AND ANALYSIS FUNCTIONS
# =============================================================================
def prepare_and_flatten_data(data_sources):
    """Loads and flattens predictions for each requested source."""
    prepared_data = {}
    print("--- Preparing and Loading Data ---")
    for source_name in data_sources:
        file_path = os.path.join(RESULTS_DIR, f"tabular_{source_name}_test.pkl")
        try:
            df = pd.read_pickle(file_path)
            print(f"Loaded: {file_path}")
        except FileNotFoundError:
            print(f"ERROR: {file_path} not found → skipped.")
            continue

        # Ground-truth arrays (“base” models) ---------------------------
        base_models_data = {}
        base_df = df[df['model_name'].str.contains('base', na=False)]
        for _, row in base_df.iterrows():
            base_models_data[row['model_name']] = np.concatenate(row['y_pred_binary'])

        # Predictive models --------------------------------------------
        models_df = df[~df['model_name'].str.contains('base', na=False)]
        for _, row in models_df.iterrows():
            model_key = row['model_name']
            if model_key not in model_translate:
                continue

            model_name_trans = model_translate[model_key]
            full_name = f"{source_name}_{model_name_trans}"

            base_to_use = model_base_map.get(model_name_trans, DEFAULT_BASE)
            if base_to_use not in base_models_data:
                print(f"WARNING: base '{base_to_use}' missing for {full_name} → skipped.")
                continue

            y_true_flat = base_models_data[base_to_use]
            y_prob_flat = np.concatenate(row['y_prob'])

            if len(y_true_flat) != len(y_prob_flat):
                print(f"WARNING: length mismatch for {full_name} → skipped.")
                continue

            prepared_data[full_name] = (y_true_flat, y_prob_flat)

    print(f"Prepared data for {len(prepared_data)} models.")
    return prepared_data

def run_comprehensive_delong_tests(prepared_data):
    """Runs DeLong’s test for every model pair that shares identical ground truth."""
    print("--- Running Pairwise DeLong Tests ---")
    all_models = sorted(m for m in prepared_data if m not in MODELS_TO_EXCLUDE)

    if not all_models:
        print("ERROR: No models left to compare.")
        return None

    grid = pd.DataFrame(index=all_models, columns=all_models, dtype=float)
    tot = len(all_models) ** 2
    done = 0

    for m1 in all_models:
        for m2 in all_models:
            done += 1
            print(f"[{done}/{tot}] {m1} vs {m2}")
            if m1 == m2:
                grid.loc[m1, m2] = np.nan
                continue

            y1, p1 = prepared_data[m1]
            y2, p2 = prepared_data[m2]

            if not np.array_equal(y1, y2):
                print("   → GT mismatch, set N/A.")
                grid.loc[m1, m2] = "N/A"
                continue

            _, _, p = delong_test(y1, p1, p2)
            grid.loc[m1, m2] = p

    return grid

def format_and_save_results(p_value_grid):
    """Pretty-print, write CSV, and show a styled HTML table in Jupyter."""
    print("--- Formatting Results ---")

    # --- MODIFICATION: Rename models for concise table display ---
    renamed_grid = p_value_grid.copy()
    new_names = [name.replace('preop_', 'p_').replace('combined_', 'c_') for name in renamed_grid.columns]
    renamed_grid.columns = new_names
    renamed_grid.index = new_names

    # Formatting function for p-values with significance stars
    def fmt(p):
        if pd.isna(p): return "nan"
        if isinstance(p, str): return p
        if p < 1e-4: return "<0.0001***"
        if p < 1e-3: return f"{p:.4f}***"
        if p < 1e-2: return f"{p:.4f}**"
        if p < 5e-2: return f"{p:.4f}*"
        return f"{p:.4f}"

    text_grid = renamed_grid.map(fmt)
    print("\nDeLong Test Results (p_ = preop, c_ = combined):\n")
    print(text_grid.to_string())

    text_grid.to_csv(OUTPUT_CSV_FILE)
    print(f"\nCSV written to {OUTPUT_CSV_FILE}")

    # --- MODIFICATION: Jupyter-friendly styled table for Google Docs ---
    # Style significant p-values with bold text AND a background color
    def highlight_significant(p):
        is_sig = isinstance(p, (float, int)) and p < 0.05
        # NEW: Return a string with both CSS properties if significant
        return 'font-weight: bold; background-color: #d4edda;' if is_sig else ''

    styled = (
        renamed_grid.style
        .map(highlight_significant)
        .format(fmt)
        .set_caption("Pairwise DeLong Test P-Values")
        .set_table_styles([
            # Set font for the entire table for better copy-paste compatibility
            {'selector': '', 'props': [('font-family', '"Times New Roman", Times, serif'),
                                       ('border-collapse', 'collapse')]},
            # Style for header cells
            {'selector': 'th', 'props': [('text-align', 'left'), ('font-weight', 'bold'),
                                         ('padding', '6px 8px'), ('border', '1px solid black')]},
            # Style for data cells
            {'selector': 'td', 'props': [('text-align', 'left'), ('padding', '6px 8px'),
                                         ('border', '1px solid black')]},
            # Style for the caption
            {'selector': 'caption', 'props': [('caption-side', 'top'),
                                              ('font-size', '1.2em'),
                                              ('font-weight', 'bold'),
                                              ('margin-bottom', '10px'),
                                              ('color', 'black')]}
        ])
    )

    try:
        # --- MODIFICATION: Create collapsible widget to show raw HTML ---
        table_html = styled.to_html()
        escaped_html = html.escape(table_html)

        widget_html = f"""
        <details style="margin-top: 20px; font-family: monospace;">
            <summary style="cursor: pointer; font-family: sans-serif; color: #0073e6;">
                Click to view raw HTML for table
            </summary>
            <pre style="background-color:#f4f4f4; border:1px solid #ddd; padding:10px; border-radius:5px; white-space: pre-wrap; word-wrap: break-word;"><code>{escaped_html}</code></pre>
        </details>
        """

        # Combine the visible table and the hidden widget for final output
        final_output = table_html + widget_html
        display(HTML(final_output))

    except NameError:
        # display() is not available outside of a Jupyter/IPython environment.
        pass

# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    data_to_compare = prepare_and_flatten_data(['preop', 'combined'])

    if data_to_compare:
        p_grid = run_comprehensive_delong_tests(data_to_compare)
        if p_grid is not None:
            format_and_save_results(p_grid)
    else:
        print("No data prepared – exiting.")


--- Preparing and Loading Data ---
Loaded: /home/server/Projects/data/AKI/results/tabular_preop_test.pkl
Loaded: /home/server/Projects/data/AKI/results/tabular_combined_test.pkl
Prepared data for 15 models.
--- Running Pairwise DeLong Tests ---
[1/144] combined_AG vs combined_AG
[2/144] combined_AG vs combined_GBT
[3/144] combined_AG vs combined_KNN
[4/144] combined_AG vs combined_LR
[5/144] combined_AG vs combined_MLP
[6/144] combined_AG vs combined_RF
[7/144] combined_AG vs preop_AG
[8/144] combined_AG vs preop_GBT
[9/144] combined_AG vs preop_KNN
[10/144] combined_AG vs preop_LR
[11/144] combined_AG vs preop_MLP
[12/144] combined_AG vs preop_RF
[13/144] combined_GBT vs combined_AG
[14/144] combined_GBT vs combined_GBT
[15/144] combined_GBT vs combined_KNN
[16/144] combined_GBT vs combined_LR
[17/144] combined_GBT vs combined_MLP
[18/144] combined_GBT vs combined_RF
[19/144] combined_GBT vs preop_AG
[20/144] combined_GBT vs preop_GBT
[21/144] combined_GBT vs preop_KNN
[22/144] combin

,c_AG,c_GBT,c_KNN,c_LR,c_MLP,c_RF,p_AG,p_GBT,p_KNN,p_LR,p_MLP,p_RF
c_AG,nan,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.0834,<0.0001***,0.5669,<0.0001***,<0.0001***,<0.0001***,0.3820
c_GBT,<0.0001***,nan,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***
c_KNN,<0.0001***,<0.0001***,nan,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***
c_LR,<0.0001***,<0.0001***,<0.0001***,nan,0.4340,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.2469,<0.0001***
c_MLP,<0.0001***,<0.0001***,<0.0001***,0.4340,nan,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.6381,<0.0001***
c_RF,0.0834,<0.0001***,<0.0001***,<0.0001***,<0.0001***,nan,<0.0001***,0.3963,<0.0001***,<0.0001***,<0.0001***,0.3863
p_AG,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,nan,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***
p_GBT,0.5669,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.3963,<0.0001***,nan,<0.0001***,<0.0001***,<0.0001***,0.7383
p_KNN,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,nan,0.0005***,<0.0001***,<0.0001***
p_LR,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,<0.0001***,0.0005***,nan,<0.0001***,<0.0001***
